# 作业 3.1：HPGe $\gamma$ 能谱刻度

## 方法

HPGe 能谱中的 full-energy peak 通常叠加在连续本底上。一次局部 peak fit 需要同时描述 signal 与本底，并从同一个拟合结果提取三个基本量：

- centroid $\mu_{ch}$：用于能量刻度；
- peak width $\sigma_{ch}$：用于探测器分辨率；
- full-energy peak area $N_{\mathrm{peak}}$：用于效率刻度。

这三个量来自同一个 peak model，所采用的本底定义也应保持一致。以下先说明原理与公式，再用两个典型 peak 演示 ROOT 中的具体做法。

### Gamma peak 与局部本底

孤立且近似对称的 gamma peak 可先写成 Gaussian signal：

$$
s(x)=H\exp\!\left[-\frac{(x-\mu)^2}{2\sigma^2}\right].
$$

$H$ 是 peak height，$\mu$ 是 centroid，$\sigma$ 是 Gaussian width。对本底较低且近似对称的孤立峰，单独用 Gaussian 已能较准确地估计 centroid；要提取 $\sigma$ 和 peak area，则应把局部本底同时放进拟合函数：

$$f(x)=s(x)+b(x).$$

在足够窄的拟合区间内，连续本底通常先用常数或直线描述：

$$b(x)=b_0+b_1(x-x_0).$$

把直线写在峰附近的参考位置 $x_0$ 上，可使 $b_0$ 直接表示峰附近的本底高度。若低能侧存在与 peak 相连的平滑 step，可加入

$$
b(x)=b_0+b_1(x-x_0)+
\frac{S}{2}\operatorname{erfc}\!\left(
\frac{x-\mu}{\sqrt{2}\sigma}\right).
$$

常数或直线适合局部平滑本底；`erfc` 项描述 full-energy peak 低能侧的 step。若拟合区间内还有邻近 peak，应增加相应的 signal 分量；若有明显低能 tail，则应使用相应的 tail model。不能仅提高本底多项式的阶数，让它吸收邻近 peak 或 peak tail。

<img src="../calibration_method/peak_model_components.png" alt="peak signal, local background, total fit and residual" style="max-width:52%;" />

图中 total model 由 Gaussian signal 和带 step 的局部本底组成；下方 residual 用于检查这个模型是否能够描述数据。

### Peak fit、面积与误差

histogram 的 bin content 是计数时，可采用 binned Poisson likelihood。第 $i$ 个 bin 的观测计数为 $n_i$、模型期望为 $\nu_i$，忽略与参数无关的常数后，

$$
-\ln L=\sum_i\left(\nu_i-n_i\ln\nu_i\right).
$$

当 bin width 不可忽略时，$\nu_i$ 应由拟合函数在该 bin 内积分得到。拟合后可画 Pearson residual

$$
r_i=\frac{n_i-\nu_i}{\sqrt{\nu_i}},
$$

检查 residual 是否在零附近无规则分布。fit status 只说明数值优化是否收敛，不能代替 residual 对模型的检验。

ROOT 的 `gaus` 使用 peak height $H$，不是面积。若 histogram 的 bin width 为 $w$，Gaussian signal 的总计数为

$$
N_{\mathrm{peak}}=\frac{H\sigma\sqrt{2\pi}}{w}.
$$

橙色区域是 total model 与同一次拟合所得本底之差，也就是 $N_{\mathrm{peak}}$：

<img src="../calibration_method/full_energy_peak_area.png" alt="signal area above the fitted local background" style="max-width:52%;" />

若拟合参数组成向量 $\boldsymbol{p}$，其 covariance matrix 为 $C$，由参数计算的物理量 $g(\boldsymbol{p})$ 在一阶近似下满足

$$
u_g^2=\boldsymbol{J}C\boldsymbol{J}^{T},
\qquad J_j=\frac{\partial g}{\partial p_j}.
$$

对 Gaussian area，令 $K=\sqrt{2\pi}/w$，则

$$
u_N^2=K^2\left[
\sigma^2u_H^2+H^2u_\sigma^2+
2H\sigma\operatorname{Cov}(H,\sigma)
\right].
$$

这里的 covariance matrix 由 joint fit 返回。它反映 peak height、width 与本底参数在拟合中的相关性；不是拟合前提供的输入。

对孤立峰还可采用 sideband subtraction。取中心 peak window 的宽度 $W_G=3\,FWHM$，左右 sideband 各宽 $1.5\,FWHM$。若 $G$、$B_L$、$B_R$ 分别为三个区域内的总计数，则

$$
N_{\mathrm{peak}}=G-B_L-B_R.
$$

更一般地，令 $\alpha=W_G/(W_L+W_R)$，则

$$
N_{\mathrm{peak}}=G-\alpha(B_L+B_R),\qquad
u_N^2\simeq G+\alpha^2(B_L+B_R),
$$

其中误差式假设三个区域互不重叠且计数服从 Poisson 分布。sideband 方法适合孤立峰和近似平滑的局部本底，可作为 fitted signal area 的交叉检查；本底有 step、邻近 peak 或 tail 时，应以完整的 peak model 为主。ROOT 中可用 `FindBin` 把由 $\mu$ 和 FWHM 定义的边界换成 bin number，再用 `Integral` 得到 $G$、$B_L$ 和 $B_R$。

<img src="../calibration_method/netcount.png" alt="peak and sideband integration regions" style="max-width:42%;" />

### 能量刻度

HPGe 的能量刻度通常使用 $^{133}$Ba、$^{137}$Cs、$^{60}$Co、$^{152}$Eu、$^{22}$Na 等标准源，覆盖所需能区；典型刻度范围为 50–3000 keV。天然本底中的 $^{40}$K 1460 keV 和 $^{208}$Tl 2614 keV gamma line 也可用于高能端检查。

下图给出 $^{152}$Eu 与 $^{133}$Ba 参考谱，用于根据能量和相对强度辨认 peak；具体计算采用后面的参考数据表。

<div style="display:flex; flex-wrap:wrap; gap:1rem; align-items:center;">
  <img src="../calibration_method/152Eu.png" alt="Eu-152 reference spectrum" style="max-width:44%; height:auto;" />
  <img src="../calibration_method/133Ba.png" alt="Ba-133 reference spectrum" style="max-width:48%; height:auto;" />
</div>

先找出两个相隔较远、指认可靠的 peak，建立粗略线性关系，再用它预测其他参考线的位置。观测 peak area 还取决于 $A(t)P_\gamma\varepsilon(E)$，因此 $P_\gamma$ 的大小只用于辅助辨认，不能直接当作实验峰面积之比。

对每条参考 gamma line，局部拟合给出 centroid $ch_i$ 及其误差，核数据给出 $E_i$。把这些量填入 `TGraphErrors` 后，先拟合

$$E(ch)=a_0+a_1ch,$$

只有 calibration residual 出现有规律的曲率时才加入二次项

$$E(ch)=a_0+a_1ch+a_2ch^2.$$

残差定义为

$$\Delta E_i=E_{i,\mathrm{ref}}-E_{\mathrm{cal}}(ch_i).$$

本数据可把 $|\Delta E|<1$ keV 作为基本检查，同时还要看 residual 是否存在系统趋势。刻度参数本身通常相关。若参数向量 $\boldsymbol{a}$ 的 covariance matrix 为 $C_a$，在给定 channel 处，刻度曲线的误差为

$$
u_{E,\mathrm{cal}}^2=\boldsymbol{j}C_a\boldsymbol{j}^{T},
$$

其中线性刻度时 $\boldsymbol{j}=(1,ch)$，二次刻度时 $\boldsymbol{j}=(1,ch,ch^2)$。若还要计入待测 peak centroid 的误差，再加上

$$\left(\frac{dE}{dch}u_{ch}\right)^2.$$

刻度关系确定后，把原 histogram 的计数映射到以 keV 为横轴的新 histogram。原 bin 的有限宽度也要随刻度关系处理；无论采用重新设定 bin edge 还是重新填充，新谱的总计数都应与原谱一致。

<img src="../calibration_method/energy.png" alt="a typical gamma-ray energy calibration curve" style="max-width:40%;" />

图示为典型能量刻度关系。最终函数阶数由本数据的 calibration residual 决定，而不是直接照用图中的函数。

### 探测器分辨率随能量的变化

peak fit 给出 channel 上的 $\sigma_{ch}$。先用能量刻度曲线的局部斜率换算

$$
\sigma_E=\left|\frac{dE}{dch}\right|\sigma_{ch},
$$

再计算

$$
FWHM=2\sqrt{2\ln2}\,\sigma_E\approx2.355\sigma_E.
$$

若暂时忽略刻度斜率的误差，

$$
u_{FWHM}\approx2.355\left|\frac{dE}{dch}\right|u_{\sigma_{ch}};
$$

需要完整误差时，仍使用前面的 covariance propagation。

在有限能区内，FWHM–$E$ 曲线可直接用经验二次式描述：

$$FWHM(E)=a_0+a_1E+a_2E^2.$$

更常用的分辨率参数化从 variance 相加出发：

$$FWHM(E)=\sqrt{A+BE+CE^2}.$$

$A$、$BE$ 和 $CE^2$ 分别概括电子学噪声、载流子统计以及随能量增长的电荷收集等贡献。直接二次式可在刻度能区内作经验 interpolation，但不保证区间外仍为正；本作业的参考结果使用平方根形式。是否需要全部三项由数据范围和 residual 判断。

<img src="../calibration_method/width.png" alt="a typical HPGe resolution curve" style="max-width:42%;" />

图示为典型 FWHM–$E$ 曲线，用来观察 HPGe peak width 随能量的总体变化。

### Full-energy peak efficiency

源在参考时刻 $t_0$ 的活度为 $A_0$。测量开始时的活度为

$$
A(t)=A_0e^{-\lambda(t-t_0)}
=A_0\,2^{-(t-t_0)/T_{1/2}},
\qquad \lambda=\frac{\ln2}{T_{1/2}}.
$$

测量期间的衰变数为

$$
N_{\mathrm{decay}}=\int_0^{t_{\mathrm{live}}}A(t+\tau)\,d\tau
=\frac{A(t)}{\lambda}\left(1-e^{-\lambda t_{\mathrm{live}}}\right)
\simeq A(t)t_{\mathrm{live}}.
$$

本作业的测量时长远短于两个源的半衰期，可以使用最后的近似。能量为 $E_\gamma$ 的 full-energy peak efficiency 为

$$
\varepsilon(E_\gamma)=
\frac{N_{\mathrm{peak}}}{N_{\mathrm{decay}}P_\gamma}
\simeq
\frac{N_{\mathrm{peak}}}{A(t)P_\gamma t_{\mathrm{live}}}.
$$

$P_\gamma$ 是每次母核衰变发射该 gamma ray 的概率，不是源活度。若各输入量相互独立，

$$
\left(\frac{u_\varepsilon}{\varepsilon}\right)^2=
\left(\frac{u_N}{N_{\mathrm{peak}}}\right)^2+
\left(\frac{u_A}{A(t)}\right)^2+
\left(\frac{u_P}{P_\gamma}\right)^2+
\left(\frac{u_t}{t_{\mathrm{live}}}\right)^2.
$$

同一标准源的活度误差会同时影响该源的全部 efficiency 点，因此这些点之间相关。若 7442 s 是 elapsed time 而不是 live time，或尚未处理 dead time、true-coincidence summing、源几何和自吸收修正，则这里得到的是该实验设置下的 apparent efficiency。

效率曲线通常在 log–log 坐标上拟合。刻度点不多时，可以在限定能区内采用

$$
\ln\varepsilon(E)=p_0+p_1\ln E+p_2(\ln E)^2.
$$

刻度点较多时，还可采用更灵活的经验形式：

$$
\varepsilon(E)=\frac{1}{E}
\sum_{i\in\{1,2,3,4,6,8\}}a_i(\ln E)^{i-1}.
$$

这类高阶参数化需要足够多的 calibration points，只适合在刻度能区内使用，并应由 residual 检查是否出现过拟合。

<img src="../calibration_method/eff.png" alt="a typical HPGe full-energy peak efficiency curve" style="max-width:42%;" />

典型 full-energy peak efficiency 在低能区先受探测器窗口、吸收和阈值影响，达到最大值后随能量升高而下降。

## 实验设置与标准源

Eurica gamma 探测阵列由 12 个 Euroball Cluster 组成，每个 Cluster 含 7 个 HPGe 晶体，标定源距探测器约 22 cm。数据已先对每个 Cluster 的 7 个晶体进行 add-back，再合并 12 个 Cluster，用于考察阵列的整体性能。装置详情见 [Installation and commissioning of EURICA – Euroball-RIKEN Cluster Array](https://www.sciencedirect.com/science/article/pii/S0168583X13003182)。

<img src="eurica.png" alt="Eurica detector array" style="max-width:42%;" />

能谱由 $^{152}$Eu 与 $^{133}$Ba 标准源测得，测量开始于 2013 年 2 月 13 日，记录时长为 7442 s。两个源的参考日期均为 1998 年 1 月 1 日；参考活度分别为 $^{152}$Eu 40.9 kBq（5%）和 $^{133}$Ba 42.2 kBq（3%）。活度衰变修正可采用 $T_{1/2}(^{152}\mathrm{Eu})=13.517$ y、$T_{1/2}(^{133}\mathrm{Ba})=3849.3$ d。

| Nuclide | $E_\gamma$ (keV) | $P_\gamma$ (%) |
| --- | ---: | ---: |
| $^{133}$Ba | 80.9979 | 34.06 |
| $^{152}$Eu | 121.7817 | 28.41 |
| $^{152}$Eu | 244.6974 | 7.55 |
| $^{133}$Ba | 276.3989 | 7.164 |
| $^{133}$Ba | 302.8508 | 18.33 |
| $^{152}$Eu | 344.2785 | 26.59 |
| $^{133}$Ba | 356.0129 | 62.05 |
| $^{152}$Eu | 778.9045 | 12.93 |
| $^{152}$Eu | 867.378 | 4.23 |
| $^{152}$Eu | 964.079 | 14.51 |
| $^{152}$Eu | 1112.076 | 13.67 |
| $^{152}$Eu | 1408.013 | 20.87 |

能量、$P_\gamma$ 和半衰期应以推荐核数据为准；可查阅 [DDEP/LNHB recommended decay data](https://www.lnhb.fr/home/nuclear-data/)。参考谱图中的强度标注来自较早资料，数值计算使用上表。

## 作业要求

### 能量刻度

1. 用 log scale 查看完整能谱。参照标准源能谱和表中的 $E_\gamma$、$P_\gamma$，先找出两个相隔较远且指认可靠的 peak，估计线性刻度关系，再据此寻找其余刻度线。
2. 对选取的刻度峰作局部 fit。孤立峰可先用 Gaussian 确定 centroid；提取 width 和 area 时使用 Gaussian 与局部本底的组合，并逐峰检查 residual。
3. 把 centroid、centroid error 和参考能量填入 `TGraphErrors`。分别作线性和二次能量刻度，画出 $\Delta E=E_{\mathrm{ref}}-E_{\mathrm{cal}}$，据 residual 选择刻度关系。
4. 用选定的刻度关系生成能量谱，并检查变换前后的总计数是否一致。

### 峰宽与能量分辨率

由同一组局部 fit 的 $\sigma_{ch}$ 计算 FWHM，画出 FWHM–$E_\gamma$ 曲线，用

$$FWHM(E)=\sqrt{A+BE+CE^2}$$

拟合，并画出 $FWHM_{\mathrm{data}}-FWHM_{\mathrm{fit}}$。根据 residual 判断曲线是否能描述数据。

### Full-energy peak efficiency（选做）

1. 从 peak fit 的 signal 分量计算 $N_{\mathrm{peak}}$，用拟合返回的 covariance matrix 传播统计误差；选择一个孤立峰，用 sideband subtraction 交叉检查面积。
2. 将两个源的活度修正到测量日期，计算测量期间的衰变数和各条 gamma line 的 apparent full-energy peak efficiency。
3. 在 log–log 坐标上拟合 efficiency–energy 曲线，并画出相对 residual。

## 实例代码

本作业使用 [gamma.root](gamma.root) 中的 `TH1F h0`。横轴是尚未刻度的 channel，范围为 0–2500，每个 bin 宽 0.2 channel。下面只对两个典型 peak 展开局部拟合，其余刻度线仍需自行完成。

<div class="code-language-switch" role="group" aria-label="Code language">
  <span>Code language:</span>
  <button type="button" data-code-language="python" aria-pressed="true">Python / PyROOT</button>
  <button type="button" data-code-language="cpp" aria-pressed="false">ROOT C++</button>
</div>

<style>
.code-language-switch { display:none; gap:.5rem; align-items:center; margin:1rem 0; }
.code-language-switch button { padding:.3rem .8rem; border:1px solid #b8b8b8; border-radius:4px; background:#fff; cursor:pointer; }
.code-language-switch button[aria-pressed="true"] { color:#fff; background:#2f6f9f; border-color:#2f6f9f; }
.pyroot-code-marker { display:none; }
.pyroot-code-marker + .highlight {
  margin:.5rem 0 1rem;
  border:1px solid #d5d5d5;
  border-radius:2px;
  background:#f7f7f7;
}
.pyroot-code-marker + .highlight pre { margin:0; padding:.75rem 1rem; overflow-x:auto; }
.pyroot-code-cell[hidden],
.jp-CodeCell .jp-Cell-inputWrapper[hidden] { display:none !important; }
</style>

<script>
document.addEventListener("DOMContentLoaded", function () {
  const buttons = document.querySelectorAll(".code-language-switch button");
  const pythonCells = Array.from(document.querySelectorAll(".pyroot-code-marker"))
    .map(function (marker) { return marker.closest(".jp-MarkdownCell"); })
    .filter(Boolean);
  pythonCells.forEach(function (cell) { cell.classList.add("pyroot-code-cell"); });
  const cppInputs = document.querySelectorAll(".jp-CodeCell .jp-Cell-inputWrapper");

  function selectLanguage(language) {
    pythonCells.forEach(function (cell) { cell.hidden = language !== "python"; });
    cppInputs.forEach(function (input) { input.hidden = language !== "cpp"; });
    buttons.forEach(function (button) {
      button.setAttribute("aria-pressed", String(button.dataset.codeLanguage === language));
    });
  }

  buttons.forEach(function (button) {
    button.addEventListener("click", function () { selectLanguage(button.dataset.codeLanguage); });
  });
  document.querySelector(".code-language-switch").style.display = "flex";
  selectLanguage("python");
});
</script>

### 读取并查看能谱

先打开文件并取得 `h0`。完整谱使用 log scale，目的是同时看见强峰和较弱峰；这一步只改变显示，不改变计数。

<div class="pyroot-code-marker"></div>

```python
import math
import ROOT

ROOT.gStyle.SetOptStat(0)

# TFile.Open 打开 ROOT 文件；Get 取得其中名为 h0 的 histogram。
input_file = ROOT.TFile.Open("gamma.root", "READ")
h0 = input_file.Get("h0")

c_spectrum = ROOT.TCanvas("c_spectrum_py", "h0", 850, 480)
c_spectrum.SetLogy()
h0.SetTitle("^{152}Eu + ^{133}Ba spectrum;channel;counts / bin")
h0.GetXaxis().SetRangeUser(40, 1300)
h0.SetMinimum(0.5)
h0.Draw("hist")
c_spectrum.Draw()
```

In [ ]:
#include "TCanvas.h"
#include "TFile.h"
#include "TF1.h"
#include "TFitResultPtr.h"
#include "TGraph.h"
#include "TH1.h"
#include "TLine.h"
#include "TMath.h"
#include "TStyle.h"
#include <algorithm>
#include <cmath>
#include <iostream>

gStyle->SetOptStat(0);

// TFile::Open 打开 ROOT 文件；Get 取得其中名为 h0 的 histogram。
auto inputFile = TFile::Open("gamma.root", "READ");
auto h0 = dynamic_cast<TH1*>(inputFile->Get("h0"));

auto cSpectrum = new TCanvas("cSpectrum", "h0", 850, 480);
cSpectrum->SetLogy();
h0->SetTitle("^{152}Eu + ^{133}Ba spectrum;channel;counts / bin");
h0->GetXaxis()->SetRangeUser(40, 1300);
h0->SetMinimum(0.5);
h0->Draw("hist");
cSpectrum->Draw();

### 选择拟合区间并设置初始值

多参数 peak model 是非线性拟合，minimizer 需要从一组初始值开始搜索。初值只需接近数据的数量级：peak maximum 减去本底可估计 $H$，peak 大致位置给出 $\mu$，初步 Gaussian fit 或峰宽给出 $\sigma$，区间两端的计数给出本底高度和斜率。合理的 parameter limits 用于排除负峰高、负峰宽等非物理解，不用于强迫拟合得到预期答案。

本例的 fit option 为 `LIRSQN`：

| option | 含义 | 本例中的作用 |
| --- | --- | --- |
| `L` | Poisson likelihood | histogram bin content 是计数 |
| `I` | 使用函数在每个 bin 内的积分 | 避免只取 bin center |
| `R` | 使用 `TF1` 定义的范围 | 只拟合当前局部 peak |
| `S` | 返回 `TFitResult` | 读取参数 covariance matrix |
| `Q` | quiet mode | 不打印完整 minimizer 过程 |
| `N` | 不保存、也不自动绘制拟合函数 | 后面自行组织 peak 与 residual 图 |

各选项的完整定义见 [ROOT `TH1::Fit` documentation](https://root.cern.ch/doc/master/classTH1.html)。

下面拟合 244.7 keV 刻度线对应的局部 peak。线性本底写成相对于 239.1 channel 的形式，使 `b0` 表示峰附近的本底高度。`SetParameter(i, value)` 中的编号与 `SetParNames` 的顺序一致。

<div class="pyroot-code-marker"></div>

```python
xmin244, xmax244 = 235.6, 242.6
f244 = ROOT.TF1("f244_py", "gaus(0)+[3]+[4]*(x-239.1)", xmin244, xmax244)
f244.SetParNames("height", "mean", "sigma", "b0", "b1")
f244.SetParameter(0, 1.8e5)  # peak maximum 减去局部本底
f244.SetParameter(1, 239.1)  # 从谱图读出的 centroid 初值
f244.SetParameter(2, 0.7)    # 由可见峰宽估计 sigma
f244.SetParameter(3, 4.0e4)  # 峰两侧计数给出 b0
f244.SetParameter(4, 0.0)    # 局部斜率先从 0 开始
f244.SetParLimits(0, 0.0, 1.0e7)
f244.SetParLimits(1, 237.0, 241.0)
f244.SetParLimits(2, 0.2, 3.0)

result244 = h0.Fit(f244, "LIRSQN")
```

In [ ]:
double xmin244 = 235.6;
double xmax244 = 242.6;
auto f244 = new TF1("f244", "gaus(0)+[3]+[4]*(x-239.1)", xmin244, xmax244);
f244->SetParNames("height", "mean", "sigma", "b0", "b1");
f244->SetParameter(0, 1.8e5);  // peak maximum 减去局部本底
f244->SetParameter(1, 239.1);  // 从谱图读出的 centroid 初值
f244->SetParameter(2, 0.7);    // 由可见峰宽估计 sigma
f244->SetParameter(3, 4.0e4);  // 峰两侧计数给出 b0
f244->SetParameter(4, 0.0);    // 局部斜率先从 0 开始
f244->SetParLimits(0, 0.0, 1.0e7);
f244->SetParLimits(1, 237.0, 241.0);
f244->SetParLimits(2, 0.2, 3.0);

// L/I/R/S/Q/N: Poisson likelihood / bin integral / function range /
// return result / quiet / do not draw automatically.
TFitResultPtr result244 = h0->Fit(f244, "LIRSQN");

### 读取 centroid、$\sigma$ 和 area

`GetParameter(1)` 和 `GetParError(1)` 分别给出 centroid 及其拟合误差；`GetParameter(2)` 给出 $\sigma$。注意 centroid error 不是 $\sigma$。先按方法部分的 Gaussian integral 计算 area。

<div class="pyroot-code-marker"></div>

```python
# GetParameter / GetParError 读取拟合参数及其标准误差。
height244 = f244.GetParameter(0)
mean244 = f244.GetParameter(1)
sigma244 = abs(f244.GetParameter(2))

# Gaussian height 和 sigma 共同决定峰面积，面积误差需要二者的 covariance。
area_factor = math.sqrt(2.0 * math.pi) / h0.GetBinWidth(1)
area244 = height244 * sigma244 * area_factor
```

In [ ]:
// GetParameter / GetParError 读取拟合参数及其标准误差。
double height244 = f244->GetParameter(0);
double mean244 = f244->GetParameter(1);
double sigma244 = std::abs(f244->GetParameter(2));

// Gaussian height 和 sigma 共同决定峰面积。
double areaFactor = std::sqrt(2.0 * TMath::Pi()) / h0->GetBinWidth(1);
double area244 = height244 * sigma244 * areaFactor;

#### 使用 covariance 计算 area error

`GetCovarianceMatrix()` 返回同一次 fit 的 parameter covariance matrix。area 同时依赖 $H$ 与 $\sigma$，因此误差中要保留 $\operatorname{Cov}(H,\sigma)$。`CovMatrixStatus()` 用来检查 covariance matrix 的质量；ROOT 返回 3 表示 full, accurate covariance matrix。

<div class="pyroot-code-marker"></div>

```python
cov244 = result244.GetCovarianceMatrix()
area_variance244 = (
    (sigma244 * area_factor)**2 * cov244[0][0]
    + (height244 * area_factor)**2 * cov244[2][2]
    + 2.0 * height244 * sigma244 * area_factor**2 * cov244[0][2]
)
area_error244 = math.sqrt(max(area_variance244, 0.0))

print("244.7 keV example")
print(f"mean   = {mean244:.4f} +/- {f244.GetParError(1):.4f}")
print(f"sigma  = {sigma244:.4f} +/- {f244.GetParError(2):.4f}")
print(f"N_peak = {area244:.0f} +/- {area_error244:.0f}")
print(f"fit status = {int(result244)}")
print(f"covariance status = {result244.CovMatrixStatus()}")
```

In [ ]:
auto cov244 = result244->GetCovarianceMatrix();
double areaVariance244 =
    std::pow(sigma244 * areaFactor, 2) * cov244(0, 0)
    + std::pow(height244 * areaFactor, 2) * cov244(2, 2)
    + 2.0 * height244 * sigma244 * areaFactor * areaFactor * cov244(0, 2);
double areaError244 = std::sqrt(std::max(areaVariance244, 0.0));

std::cout << "244.7 keV example\n"
          << "mean   = " << mean244 << " +/- " << f244->GetParError(1) << "\n"
          << "sigma  = " << sigma244 << " +/- " << f244->GetParError(2) << "\n"
          << "N_peak = " << area244 << " +/- " << areaError244 << "\n"
          << "fit status = " << static_cast<int>(result244) << "\n"
          << "covariance status = " << result244->CovMatrixStatus() << "\n";

### 计算并绘制 residual

逐 bin 计算模型期望和 Pearson residual。因为拟合使用了 `I`，这里也对函数做 bin integral，再除以 bin width 得到该 bin 的期望高度。

<div class="pyroot-code-marker"></div>

```python
residual244 = ROOT.TGraph()
for point, bin_number in enumerate(
    range(h0.FindBin(xmin244), h0.FindBin(xmax244) + 1)
):
    low = h0.GetBinLowEdge(bin_number)
    width = h0.GetBinWidth(bin_number)
    expected = f244.Integral(low, low + width) / width
    observed = h0.GetBinContent(bin_number)
    residual244.SetPoint(
        point, h0.GetBinCenter(bin_number),
        (observed - expected) / math.sqrt(expected)
    )
```

In [ ]:
auto residual244 = new TGraph();
int point244 = 0;
for (int bin = h0->FindBin(xmin244); bin <= h0->FindBin(xmax244); ++bin) {
    double low = h0->GetBinLowEdge(bin);
    double width = h0->GetBinWidth(bin);
    double expected = f244->Integral(low, low + width) / width;
    double observed = h0->GetBinContent(bin);
    residual244->SetPoint(point244++, h0->GetBinCenter(bin),
                          (observed - expected) / std::sqrt(expected));
}

将局部 peak 与总拟合函数画在上方，residual 画在下方。显示用的 histogram 是 `h0` 的 clone，不会改变原始能谱。

<div class="pyroot-code-marker"></div>

```python
c244 = ROOT.TCanvas("c244_py", "244.7 keV example", 720, 650)
c244.Divide(1, 2)
c244.cd(1)
h244_view = h0.Clone("h244_view_py")
h244_view.GetXaxis().SetRangeUser(xmin244, xmax244)
h244_view.SetTitle("244.7 keV example;channel;counts / bin")
h244_view.Draw("E")
f244.SetLineColor(ROOT.kBlue + 1)
f244.Draw("same")
c244.cd(2)
residual244.SetTitle("Fit residual;channel;(n-#mu)/#sqrt{#mu}")
residual244.SetMarkerStyle(20)
residual244.Draw("AP")
zero244 = ROOT.TLine(xmin244, 0.0, xmax244, 0.0)
zero244.SetLineStyle(2)
zero244.Draw()
c244.Draw()
```

In [ ]:
auto c244 = new TCanvas("c244", "244.7 keV example", 720, 650);
c244->Divide(1, 2);
c244->cd(1);
auto h244View = static_cast<TH1*>(h0->Clone("h244View"));
h244View->GetXaxis()->SetRangeUser(xmin244, xmax244);
h244View->SetTitle("244.7 keV example;channel;counts / bin");
h244View->Draw("E");
f244->SetLineColor(kBlue + 1);
f244->Draw("same");
c244->cd(2);
residual244->SetTitle("Fit residual;channel;(n-#mu)/#sqrt{#mu}");
residual244->SetMarkerStyle(20);
residual244->Draw("AP");
auto zero244 = new TLine(xmin244, 0.0, xmax244, 0.0);
zero244->SetLineStyle(2);
zero244->Draw();
c244->Draw();

### 低能侧 step 的处理

867.4 keV 示例的低能侧本底高于高能侧，因此在相同 Gaussian + linear background 上加入 `erfc` step。参数 `step` 的初值由峰两侧本底高度差估计；其余步骤与前一个 peak 相同。

<div class="pyroot-code-marker"></div>

```python
# 拟合新峰前恢复 h0 的完整 channel 范围。
h0.GetXaxis().SetRangeUser(0.0, 2500.0)
xmin867, xmax867 = 758.2, 767.2
model867 = (
    "gaus(0)+[3]+[4]*(x-762.7)"
    "+[5]*0.5*TMath::Erfc((x-[1])/(sqrt(2)*[2]))"
)
f867 = ROOT.TF1("f867_py", model867, xmin867, xmax867)
f867.SetParNames("height", "mean", "sigma", "b0", "b1", "step")
f867.SetParameter(0, 4.5e4)  # peak height
f867.SetParameter(1, 762.7)  # centroid
f867.SetParameter(2, 0.8)    # sigma
f867.SetParameter(3, 6.0e3)  # peak 附近的连续本底
f867.SetParameter(4, 0.0)    # 局部斜率
f867.SetParameter(5, 1.5e3)  # 左右本底高度差给出 step 初值
f867.SetParLimits(0, 0.0, 1.0e7)
f867.SetParLimits(1, 760.0, 765.0)
f867.SetParLimits(2, 0.2, 3.0)
f867.SetParLimits(5, 0.0, 1.0e6)

result867 = h0.Fit(f867, "LIRSQN")
```

In [ ]:
// 拟合新峰前恢复 h0 的完整 channel 范围。
h0->GetXaxis()->SetRangeUser(0.0, 2500.0);
double xmin867 = 758.2;
double xmax867 = 767.2;
auto f867 = new TF1(
    "f867",
    "gaus(0)+[3]+[4]*(x-762.7)+[5]*0.5*TMath::Erfc((x-[1])/(sqrt(2)*[2]))",
    xmin867, xmax867);
f867->SetParNames("height", "mean", "sigma", "b0", "b1", "step");
f867->SetParameter(0, 4.5e4);  // peak height
f867->SetParameter(1, 762.7);  // centroid
f867->SetParameter(2, 0.8);    // sigma
f867->SetParameter(3, 6.0e3);  // peak 附近的连续本底
f867->SetParameter(4, 0.0);    // 局部斜率
f867->SetParameter(5, 1.5e3);  // 左右本底高度差给出 step 初值
f867->SetParLimits(0, 0.0, 1.0e7);
f867->SetParLimits(1, 760.0, 765.0);
f867->SetParLimits(2, 0.2, 3.0);
f867->SetParLimits(5, 0.0, 1.0e6);

TFitResultPtr result867 = h0->Fit(f867, "LIRSQN");

### 第二个 peak 的参数与面积

拟合完成后，仍只从 Gaussian signal 的 $H$ 和 $\sigma$ 计算 full-energy peak area；step 与直线属于本底。面积误差使用同一次拟合返回的 covariance matrix。

<div class="pyroot-code-marker"></div>

```python
height867 = f867.GetParameter(0)
mean867 = f867.GetParameter(1)
sigma867 = abs(f867.GetParameter(2))
area867 = height867 * sigma867 * area_factor
```

In [ ]:
double height867 = f867->GetParameter(0);
double mean867 = f867->GetParameter(1);
double sigma867 = std::abs(f867->GetParameter(2));
double area867 = height867 * sigma867 * areaFactor;

`CovMatrixStatus()` 的判断与第一个 peak 相同；下面传播 $H$ 和 $\sigma$ 的误差并输出结果。

<div class="pyroot-code-marker"></div>

```python
cov867 = result867.GetCovarianceMatrix()
area_variance867 = (
    (sigma867 * area_factor)**2 * cov867[0][0]
    + (height867 * area_factor)**2 * cov867[2][2]
    + 2.0 * height867 * sigma867 * area_factor**2 * cov867[0][2]
)
area_error867 = math.sqrt(max(area_variance867, 0.0))

print("867.4 keV example")
print(f"mean   = {mean867:.4f} +/- {f867.GetParError(1):.4f}")
print(f"sigma  = {sigma867:.4f} +/- {f867.GetParError(2):.4f}")
print(f"N_peak = {area867:.0f} +/- {area_error867:.0f}")
print(f"fit status = {int(result867)}")
print(f"covariance status = {result867.CovMatrixStatus()}")
```

In [ ]:
auto cov867 = result867->GetCovarianceMatrix();
double areaVariance867 =
    std::pow(sigma867 * areaFactor, 2) * cov867(0, 0)
    + std::pow(height867 * areaFactor, 2) * cov867(2, 2)
    + 2.0 * height867 * sigma867 * areaFactor * areaFactor * cov867(0, 2);
double areaError867 = std::sqrt(std::max(areaVariance867, 0.0));

std::cout << "867.4 keV example\n"
          << "mean   = " << mean867 << " +/- " << f867->GetParError(1) << "\n"
          << "sigma  = " << sigma867 << " +/- " << f867->GetParError(2) << "\n"
          << "N_peak = " << area867 << " +/- " << areaError867 << "\n"
          << "fit status = " << static_cast<int>(result867) << "\n"
          << "covariance status = " << result867->CovMatrixStatus() << "\n";

最后按同样的方法计算第二个 peak 的 residual。加入 step 后仍需由 residual 判断模型是否足够，不能只看 fit status。

<div class="pyroot-code-marker"></div>

```python
residual867 = ROOT.TGraph()
for point, bin_number in enumerate(
    range(h0.FindBin(xmin867), h0.FindBin(xmax867) + 1)
):
    low = h0.GetBinLowEdge(bin_number)
    width = h0.GetBinWidth(bin_number)
    expected = f867.Integral(low, low + width) / width
    observed = h0.GetBinContent(bin_number)
    residual867.SetPoint(
        point, h0.GetBinCenter(bin_number),
        (observed - expected) / math.sqrt(expected)
    )
```

In [ ]:
auto residual867 = new TGraph();
int point867 = 0;
for (int bin = h0->FindBin(xmin867); bin <= h0->FindBin(xmax867); ++bin) {
    double low = h0->GetBinLowEdge(bin);
    double width = h0->GetBinWidth(bin);
    double expected = f867->Integral(low, low + width) / width;
    double observed = h0->GetBinContent(bin);
    residual867->SetPoint(point867++, h0->GetBinCenter(bin),
                          (observed - expected) / std::sqrt(expected));
}

绘图结构与第一个 peak 相同，便于直接比较两种本底模型。

<div class="pyroot-code-marker"></div>

```python
c867 = ROOT.TCanvas("c867_py", "867.4 keV example", 720, 650)
c867.Divide(1, 2)
c867.cd(1)
h867_view = h0.Clone("h867_view_py")
h867_view.GetXaxis().SetRangeUser(xmin867, xmax867)
h867_view.SetTitle("867.4 keV example;channel;counts / bin")
h867_view.Draw("E")
f867.SetLineColor(ROOT.kBlue + 1)
f867.Draw("same")
c867.cd(2)
residual867.SetTitle("Fit residual;channel;(n-#mu)/#sqrt{#mu}")
residual867.SetMarkerStyle(20)
residual867.Draw("AP")
zero867 = ROOT.TLine(xmin867, 0.0, xmax867, 0.0)
zero867.SetLineStyle(2)
zero867.Draw()
c867.Draw()
```

In [ ]:
auto c867 = new TCanvas("c867", "867.4 keV example", 720, 650);
c867->Divide(1, 2);
c867->cd(1);
auto h867View = static_cast<TH1*>(h0->Clone("h867View"));
h867View->GetXaxis()->SetRangeUser(xmin867, xmax867);
h867View->SetTitle("867.4 keV example;channel;counts / bin");
h867View->Draw("E");
f867->SetLineColor(kBlue + 1);
f867->Draw("same");
c867->cd(2);
residual867->SetTitle("Fit residual;channel;(n-#mu)/#sqrt{#mu}");
residual867->SetMarkerStyle(20);
residual867->Draw("AP");
auto zero867 = new TLine(xmin867, 0.0, xmax867, 0.0);
zero867->SetLineStyle(2);
zero867->Draw();
c867->Draw();

## 参考结果

以下结果用于完成作业后的核对。两个实例 peak 的局部 fit 给出：

| gamma line | local model | centroid (channel) | $\sigma$ (channel) | $N_{\mathrm{peak}}$ |
| --- | --- | ---: | ---: | ---: |
| 244.697 keV | Gaussian + linear background | $239.1914\pm0.0008$ | $0.6519\pm0.0008$ | $(1.4299\pm0.0018)\times10^6$ |
| 867.378 keV | Gaussian + linear background + `erfc` step | $762.6353\pm0.0022$ | $0.8199\pm0.0014$ | $(4.8050\pm0.0088)\times10^5$ |

两个 fit 的 status 均为 0，covariance status 均为 3。867 keV peak 的 residual 仍可见小的系统结构，说明加入 step 后的模型也只是本例采用的局部近似。

使用全部刻度线后，一次能量刻度给出

$$E_\gamma\;(\mathrm{keV})\approx-39.943+1.189802\,ch,$$

最大绝对 calibration residual 约为 0.10 keV；二次项没有改善本数据的最大 residual。

<style>
.result-grid { display:grid; grid-template-columns:repeat(2,minmax(0,1fr)); gap:1rem; margin:1rem 0 1.5rem; }
.result-grid figure { margin:0; padding:.6rem; border:1px solid #ddd; background:#fff; }
.result-grid img { display:block; width:100%; height:auto; }
.result-grid figcaption { margin-top:.5rem; font-size:.92rem; line-height:1.4; }
@media (max-width:720px) { .result-grid { grid-template-columns:1fr; } }
</style>

### 能量刻度

<div class="result-grid">
  <figure>
    <img src="reference_calibrated_spectrum.png" alt="calibrated gamma spectrum" />
    <figcaption>将最终刻度关系应用于原始 histogram 后得到的能量谱；变换前后总计数一致。</figcaption>
  </figure>
  <figure>
    <img src="reference_energy_calibration.png" alt="energy calibration and residuals" />
    <figcaption>线性与二次刻度关系及其 calibration residual。残差决定是否需要二次项。</figcaption>
  </figure>
</div>

### 峰宽与效率

<div class="result-grid">
  <figure>
    <img src="reference_fwhm.png" alt="FWHM versus energy and residuals" />
    <figcaption>FWHM–$E_\gamma$ 曲线及 residual；这里使用 $\sqrt{A+BE+CE^2}$。</figcaption>
  </figure>
  <figure>
    <img src="reference_efficiency.png" alt="apparent full-energy peak efficiency and residuals" />
    <figcaption>Apparent full-energy peak efficiency 及相对 residual，横纵坐标均为 log scale。</figcaption>
  </figure>
</div>